In [1]:
from pyspark.sql import SparkSession

#Must set this env variable to avoid warnigns
import os
os.environ["PYARROW_IGNORE_TIMEZONE"] = '1'

import pyspark.pandas as ps

# Initialize Spark Session
spark = SparkSession.builder \
.appName("Transform and Apply in Pandas API on Spark") \
.config("spark.sql.ansi.enabled", "false") \
.config("spark.executorEnv.PYARROW_IGNORE_TIMEZONE", "1") \
.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-pattern-layout-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/06 14:39:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
#1. Create a pandas-on-Spark DataFrame

ps_df = ps.DataFrame({
    "id": [1,2,3,4,5],
    "name": ["Alice", "Bob", "Charlie", "David", "Emma"],
    "age": [25,30,35,40,45],
    "salary": [50000,60000,75000,80000,120000]
})

In [3]:
#1 Using transform() for element-wise operations
ps_df["age_in_10_years"] = ps_df["age"].transform(lambda x: x + 10)
print("\n Dataframe after transform (age + 10 years):")
print(ps_df)


 Dataframe after transform (age + 10 years):


   id     name  age  salary  age_in_10_years
0   1    Alice   25   50000               35
1   2      Bob   30   60000               40
2   3  Charlie   35   75000               45
3   4    David   40   80000               50
4   5     Emma   45  120000               55


In [4]:
# 2. Using apply() on columns
# Define a custom function to categorize salary levels
def categorize_salary(salary):
    if salary < 60000:
        return "Low"
    elif salary < 10000:
        return "Medium"
    else:
        return "High"
        

In [5]:
ps_df["salary_category"] = ps_df["salary"].apply(categorize_salary)
print("\n DataFrame after apply (Salary Category):")
print(ps_df)


 DataFrame after apply (Salary Category):


   id     name  age  salary  age_in_10_years salary_category
0   1    Alice   25   50000               35             Low
1   2      Bob   30   60000               40            High
2   3  Charlie   35   75000               45            High
3   4    David   40   80000               50            High
4   5     Emma   45  120000               55            High


In [7]:
# 3. Using apply() on rows
# define a function to format name and age
def format_row(row):
    return f"{row['name']} ({row['age']} years old)"

In [9]:
ps_df["name_with_age"] = ps_df.apply(format_row, axis=1)
print("\nDataFrame after apply on rows (name_with_age):")
print(ps_df)

/opt/spark/python/pyspark/pandas/utils.py:1017: PandasAPIOnSparkAdviceWarning: If the type hints is not specified for `apply`, it is expensive to infer the data type internally.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)



DataFrame after apply on rows (name_with_age):
   id     name  age  salary  age_in_10_years salary_category           name_with_age
0   1    Alice   25   50000               35             Low    Alice (25 years old)
1   2      Bob   30   60000               40            High      Bob (30 years old)
2   3  Charlie   35   75000               45            High  Charlie (35 years old)
3   4    David   40   80000               50            High    David (40 years old)
4   5     Emma   45  120000               55            High     Emma (45 years old)


In [10]:
spark.stop()